::::{tip} Laboratory Task 3
:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

<b>Instruction:</b> Perform a forward and backward propagation in python using the inputs from <b>Laboratory Task 2</b>.
:::

```python
x = np.array([1, 0, 1])
y = np.array([1])

# use relu as the activation function.

# learning rate
lr = 0.001
```
::::

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

This picks up right where Laboratory Task 2 left off. The forward pass is exactly the same computation as
before, just written with matrices this time instead of doing each $Z$ and $H$ by hand. The new part is the
backward pass, where the error computed in Task 2 is used to figure out how much each weight should change.
:::

<h4 style="font-family: Times New Roman"><strong>Step 1: Set Up the Inputs and Weights</strong></h4>

In [1]:
import numpy as np

# given input and target
x = np.array([1, 0, 1], dtype=float)
y = np.array([1], dtype=float)

# learning rate
lr = 0.001

# hidden layer weights, arranged as a (3 inputs x 2 hidden units) matrix
# column 0 -> hidden unit 1, column 1 -> hidden unit 2
Wh = np.array([
    [0.2, -0.3],   # weights coming from x1 (w11, w12)
    [0.4,  0.1],   # weights coming from x2 (w13, w14)
    [-0.5, 0.2]    # weights coming from x3 (w15, w16)
])
theta_h = np.array([-0.4, 0.2])   # theta1, theta2

# output layer weights, (2 hidden units x 1 output)
Wo = np.array([
    [-0.3],   # w21
    [-0.2]    # w22
])
theta_o = np.array([0.1])   # theta3

print("Wh =\n", Wh)
print("theta_h =", theta_h)
print("Wo =\n", Wo)
print("theta_o =", theta_o)

Wh =
 [[ 0.2 -0.3]
 [ 0.4  0.1]
 [-0.5  0.2]]
theta_h = [-0.4  0.2]
Wo =
 [[-0.3]
 [-0.2]]
theta_o = [0.1]


<h4 style="font-family: Times New Roman"><strong>Step 2: ReLU and Its Derivative</strong></h4>

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

For the forward pass, only $f(z) = max(0, z)$ is needed, same as in Laboratory Task 2. But for the backward
pass, the <em>derivative</em> of ReLU is also needed, since backpropagation relies on the chain rule.
:::

$$f'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \le 0 \end{cases}$$

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

Basically, if a unit was "on" during the forward pass (output greater than 0), the gradient flows through it.
If it was "off" (clipped to 0), no gradient flows through it at all.
:::

In [2]:
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

<h4 style="font-family: Times New Roman"><strong>Step 3: Forward Pass</strong></h4>

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

Same idea as Laboratory Task 2, just written as matrix operations so it generalizes past just 2 hidden units.
:::

$$Z_h = xW_h + \theta_h \qquad H = f(Z_h)$$
$$Z_o = HW_o + \theta_o \qquad \hat{y} = f(Z_o)$$

In [3]:
# hidden layer
Zh = x @ Wh + theta_h
H = relu(Zh)

# output layer
Zo = H @ Wo + theta_o
y_hat = relu(Zo)

print("Zh =", Zh)
print("H  =", H)
print("Zo =", Zo)
print("y_hat =", y_hat)

Zh = [-0.7  0.1]
H  = [0.  0.1]
Zo = [0.08]
y_hat = [0.08]


<h4 style="font-family: Times New Roman"><strong>Step 4: Compute the Error</strong></h4>

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

Using the squared error loss:
:::

$$E = \frac{1}{2}(y - \hat{y})^2$$

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

This should match the value from Laboratory Task 2, since nothing about the forward pass has changed.
:::

In [4]:
error = 0.5 * np.sum((y - y_hat) ** 2)
print("error =", error)

error = 0.4232


<h4 style="font-family: Times New Roman"><strong>Step 5: Backward Pass &mdash; Output Layer</strong></h4>

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

The output error signal is the derivative of the loss with respect to $Z_o$. Since $E = \frac{1}{2}(y-\hat{y})^2$,
its derivative with respect to $\hat{y}$ is $(\hat{y} - y)$. Multiplying that by $f'(Z_o)$ (the ReLU
derivative) gives the delta for the output unit:
:::

$$\delta_o = (\hat{y} - y) \cdot f'(Z_o)$$

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

From there, the gradients for the output weights and bias are:
:::

$$\frac{\partial E}{\partial W_o} = H^T \delta_o \qquad \frac{\partial E}{\partial \theta_o} = \delta_o$$

In [5]:
delta_o = (y_hat - y) * relu_deriv(Zo)

dWo = np.outer(H, delta_o)
dtheta_o = delta_o

print("delta_o  =", delta_o)
print("dWo      =\n", dWo)
print("dtheta_o =", dtheta_o)

delta_o  = [-0.92]
dWo      =
 [[-0.   ]
 [-0.092]]
dtheta_o = [-0.92]


<h4 style="font-family: Times New Roman"><strong>Step 6: Backward Pass &mdash; Hidden Layer</strong></h4>

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

The error now needs to be propagated one layer back. Each hidden unit's delta depends on how much it
contributed to the output error (through $W_o$), scaled by its own ReLU derivative:
:::

$$\delta_h = (\delta_o W_o^T) \cdot f'(Z_h)$$

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

Then the gradients for the hidden weights and biases:
:::

$$\frac{\partial E}{\partial W_h} = x^T \delta_h \qquad \frac{\partial E}{\partial \theta_h} = \delta_h$$

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

Notice that $Z_1 = -0.7$ was negative, so $f'(Z_1) = 0$. That means whatever gradient tries to flow back to
that unit gets zeroed out completely &mdash; it won't update at all this round.
:::

In [6]:
delta_h = (delta_o @ Wo.T) * relu_deriv(Zh)

dWh = np.outer(x, delta_h)
dtheta_h = delta_h

print("delta_h  =", delta_h)
print("dWh      =\n", dWh)
print("dtheta_h =", dtheta_h)

delta_h  = [0.    0.184]
dWh      =
 [[0.    0.184]
 [0.    0.   ]
 [0.    0.184]]
dtheta_h = [0.    0.184]


<h4 style="font-family: Times New Roman"><strong>Step 7: Update the Weights</strong></h4>

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

Standard gradient descent update rule, using the learning rate given in the task ($\alpha = 0.001$):
:::

$$W = W - \alpha \frac{\partial E}{\partial W} \qquad \theta = \theta - \alpha \frac{\partial E}{\partial \theta}$$

In [7]:
Wh_new = Wh - lr * dWh
theta_h_new = theta_h - lr * dtheta_h

Wo_new = Wo - lr * dWo
theta_o_new = theta_o - lr * dtheta_o

print("Updated Wh =\n", Wh_new)
print("Updated theta_h =", theta_h_new)
print("Updated Wo =\n", Wo_new)
print("Updated theta_o =", theta_o_new)

Updated Wh =
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]
Updated theta_h = [-0.4       0.199816]
Updated Wo =
 [[-0.3     ]
 [-0.199908]]
Updated theta_o = [0.10092]


<h4 style="font-family: Times New Roman"><strong>Summary</strong></h4>

<center>
<table style="display: inline-block; font-family: Times New Roman">
<tr><th>Weight</th><th>Before</th><th>After</th></tr>
<tr><td>$w_{11}$</td><td>0.2</td><td>0.2</td></tr>
<tr><td>$w_{12}$</td><td>-0.3</td><td>-0.300184</td></tr>
<tr><td>$w_{13}$</td><td>0.4</td><td>0.4</td></tr>
<tr><td>$w_{14}$</td><td>0.1</td><td>0.1</td></tr>
<tr><td>$w_{15}$</td><td>-0.5</td><td>-0.5</td></tr>
<tr><td>$w_{16}$</td><td>0.2</td><td>0.199816</td></tr>
<tr><td>$\theta_1$</td><td>-0.4</td><td>-0.4</td></tr>
<tr><td>$\theta_2$</td><td>0.2</td><td>0.199816</td></tr>
<tr><td>$w_{21}$</td><td>-0.3</td><td>-0.3</td></tr>
<tr><td>$w_{22}$</td><td>-0.2</td><td>-0.199908</td></tr>
<tr><td>$\theta_3$</td><td>0.1</td><td>0.10092</td></tr>
</table>
</center>

:::{div}
:style: "font-family:Times New Roman; text-align:justify; font-size:15px"

A few things worth pointing out:
:::

<ul style="font-family:Times New Roman; font-size:15px">
    <li>Any weight connected to the first hidden unit ($w_{11}$, $w_{13}$, $w_{15}$, $\theta_1$) didn't change
    at all. That's because $Z_1$ was negative during the forward pass, so ReLU shut that unit off, and once a
    unit is off it stops passing gradient backward too.</li>
    <li>The weights connected to the second hidden unit did move, since that unit was active.</li>
    <li>With a learning rate this small (0.001), the updates are tiny after just one pass. It would take many
    iterations of this same process for the network to actually learn something useful, but this is exactly
    what happens on every single step of training.</li>
</ul>